In [69]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder

In [70]:
dataset = pd.read_csv("../data_set/Zomato Restaurant Dataset.csv")
dataset.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [71]:
dataset.describe()

,Restaurant ID,Country Code,Longitude,Latitude,Average Cost for two,Price range,Aggregate rating,Votes
count,9.551000e+03,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000
mean,9.051128e+06,18.365616,64.126574,25.854381,1199.210763,1.804837,2.666370,156.909748
std,8.791521e+06,56.750546,41.467058,11.007935,16121.183073,0.905609,1.516378,430.169145
min,5.300000e+01,1.000000,-157.948486,-41.330428,0.000000,1.000000,0.000000,0.000000
25%,3.019625e+05,1.000000,77.081343,28.478713,250.000000,1.000000,2.500000,5.000000
50%,6.004089e+06,1.000000,77.191964,28.570469,400.000000,2.000000,3.200000,31.000000
75%,1.835229e+07,1.000000,77.282006,28.642758,700.000000,2.000000,3.700000,131.000000
max,1.850065e+07,216.000000,174.832089,55.976980,800000.000000,4.000000,4.900000,10934.000000


In [72]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   str    
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   str    
 4   Address               9551 non-null   str    
 5   Locality              9551 non-null   str    
 6   Locality Verbose      9551 non-null   str    
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   str    
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   str    
 12  Has Table booking     9551 non-null   str    
 13  Has Online delivery   9551 non-null   str    
 14  Is delivering now     9551 non-null   str    
 15  Switch to order menu  9551 non-n

In [73]:
dataset.shape

(9551, 21)

In [74]:
num_col = dataset.select_dtypes(include=[np.number]).columns
print(num_col)

Index(['Restaurant ID', 'Country Code', 'Longitude', 'Latitude',
       'Average Cost for two', 'Price range', 'Aggregate rating', 'Votes'],
      dtype='str')


In [75]:
col_trans = ColumnTransformer(
    transformers=[
        (
            "num_mode",
            SimpleImputer(strategy="most_frequent"),
            num_col,
        ),
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,  # Stops scikit-learn from prefixing 'step__' to columns
).set_output(transform="pandas")

dataset = col_trans.fit_transform(dataset)

In [76]:
# cols_to_remove = ["Restaurant ID", "Votes"]
# num_col_new = num_col.difference(cols_to_remove)
# num_col_new

In [77]:
# Skewness of a single column
skew_val = dataset["Votes"].skew()
print(f"Skewness: {skew_val:.2f}")

# Skewness of all numeric columns in a dataset
all_skew = dataset.select_dtypes(include=[np.number]).skew()
print(all_skew)

Skewness: 8.81
Restaurant ID            0.061570
Country Code             3.043965
Longitude               -2.807328
Latitude                -3.081635
Average Cost for two    35.477915
Price range              0.889618
Aggregate rating        -0.954130
Votes                    8.807637
dtype: float64


In [78]:
# Fixing Positive (Right) Skewness
# Best for strongly right-skewed data. Use np.log1p(x) (which computes log(1+x))
# Apply Log1p transformation directly
dataset["Votes"] = np.log1p(dataset["Votes"])
skew_val = dataset["Votes"].skew()
print(f"Skewness: {skew_val:.2f}")

Skewness: -0.03


In [79]:
# # Fixing Negative (Left) Skewness
# # K is max + 1 to keep values strictly positive
# max_val = dataset["Average Cost for two"].max() + 1
# dataset["Average Cost for two"] = np.log(max_val - dataset["Average Cost for two"])

# skew_val = dataset["Average Cost for two"].skew()
# print(f"Skewness: {skew_val:.2f}")

In [80]:
# cols_to_drop = ["ID", "Unused_Col", "Target"]
# dataset = dataset.drop(cols_to_drop, axis=1)

In [81]:
# Set threshold dynamically at the 75th percentile
threshold = dataset["Aggregate rating"].quantile(0.75)
print(f"Calculated Threshold (75th percentile): {threshold:.2f}")

# Create binary target column
dataset["high_rated"] = (
    dataset["Aggregate rating"] >= threshold
).astype(int)

Calculated Threshold (75th percentile): 3.70


In [82]:
# 2. Create 'num_cuisines' feature by counting commas in Cuisines string
dataset["num_cuisines"] = dataset['Cuisines'].apply(lambda x:len(str(x).split(",")))

In [83]:
# 3. List of columns to drop as instructed
cols_to_drop = [
    "Restaurant ID",
    "Restaurant Name",
    "Address",
    "Locality",
    "Locality Verbose",
    "Switch to order menu",
    "Aggregate rating",  # Data leakage column
    "Rating color",  # Data leakage column
    "Rating text",  # Data leakage column
    "Cuisines",  # Replaced by num_cuisines
]

# Drop the columns
dataset = dataset.drop(columns=cols_to_drop, errors="ignore")

In [84]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Country Code          9551 non-null   float64
 1   Longitude             9551 non-null   float64
 2   Latitude              9551 non-null   float64
 3   Average Cost for two  9551 non-null   float64
 4   Price range           9551 non-null   float64
 5   Votes                 9551 non-null   float64
 6   City                  9551 non-null   str    
 7   Currency              9551 non-null   str    
 8   Has Table booking     9551 non-null   str    
 9   Has Online delivery   9551 non-null   str    
 10  Is delivering now     9551 non-null   str    
 11  high_rated            9551 non-null   int64  
 12  num_cuisines          9551 non-null   int64  
dtypes: float64(6), int64(2), str(5)
memory usage: 970.2 KB


In [85]:
dataset.head()

,Country Code,Longitude,Latitude,Average Cost for two,Price range,Votes,City,Currency,Has Table booking,Has Online delivery,Is delivering now,high_rated,num_cuisines
0,162.0,121.027535,14.565443,1100.0,3.0,5.752573,Makati City,Botswana Pula(P),Yes,No,No,1,3
1,162.0,121.014101,14.553708,1200.0,3.0,6.383507,Makati City,Botswana Pula(P),Yes,No,No,1,1
2,162.0,121.056831,14.581404,4000.0,4.0,5.602119,Mandaluyong City,Botswana Pula(P),Yes,No,No,1,4
3,162.0,121.056475,14.585318,1500.0,4.0,5.902633,Mandaluyong City,Botswana Pula(P),No,No,No,1,2
4,162.0,121.057508,14.584450,1500.0,4.0,5.438079,Mandaluyong City,Botswana Pula(P),Yes,No,No,1,2


In [86]:
le_city = LabelEncoder()
le_currency = LabelEncoder()
dataset['City'] = le_city.fit_transform(dataset['City'])
dataset['Currency'] = le_currency.fit_transform(dataset['Currency'])

yes_no_cols = ["Has Table booking", "Has Online delivery", "Is delivering now"]
for col in yes_no_cols:
    dataset[col] = dataset[col].map({"Yes":1,"No":0})

In [87]:
from sklearn.preprocessing import StandardScaler

feature_col = ["Average Cost for two","Price range","Votes"]
scaler = ColumnTransformer(
    transformers=[
        (
            "feature",
            StandardScaler().set_output(transform="pandas"),
            feature_col,
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

dataset = scaler.fit_transform(dataset)

In [88]:
dataset.head()

,Average Cost for two,Price range,Votes,Country Code,Longitude,Latitude,City,Currency,Has Table booking,Has Online delivery,Is delivering now,high_rated,num_cuisines
0,-0.006154,1.319803,1.188688,162.0,121.027535,14.565443,73,0,1,0,0,1,3
1,0.000049,1.319803,1.498433,162.0,121.014101,14.553708,73,0,1,0,0,1,1
2,0.173743,2.424090,1.114826,162.0,121.056831,14.581404,75,0,1,0,0,1,4
3,0.018659,2.424090,1.262357,162.0,121.056475,14.585318,75,0,0,0,0,1,2
4,0.018659,2.424090,1.034294,162.0,121.057508,14.584450,75,0,1,0,0,1,2


In [90]:
X = dataset.drop("high_rated",axis=1)
y = dataset["high_rated"]

In [91]:
X.head()

,Average Cost for two,Price range,Votes,Country Code,Longitude,Latitude,City,Currency,Has Table booking,Has Online delivery,Is delivering now,num_cuisines
0,-0.006154,1.319803,1.188688,162.0,121.027535,14.565443,73,0,1,0,0,3
1,0.000049,1.319803,1.498433,162.0,121.014101,14.553708,73,0,1,0,0,1
2,0.173743,2.424090,1.114826,162.0,121.056831,14.581404,75,0,1,0,0,4
3,0.018659,2.424090,1.262357,162.0,121.056475,14.585318,75,0,0,0,0,2
4,0.018659,2.424090,1.034294,162.0,121.057508,14.584450,75,0,1,0,0,2


In [92]:
y.value_counts()

high_rated
0    7009
1    2542
Name: count, dtype: int64

In [94]:
y.value_counts(normalize=True) * 100

high_rated
0    73.384986
1    26.615014
Name: proportion, dtype: float64

In [95]:
# train and test data split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [96]:
X_test.shape

(1911, 12)

In [97]:
# 2. Scale features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [108]:
from sklearn.decomposition import PCA
# 3. Apply PCA
# Option A: Keep specific number of components (e.g., n_components=2 or 5)
pca = PCA(n_components=9)
X_pca = pca.fit_transform(X_scaled)

# Option B: Keep components that explain a percentage of total variance (e.g., 95%)
# pca = PCA(n_components=0.95)
# X_pca = pca.fit_transform(X_scaled)

In [109]:
# 4. Convert output back to a pandas DataFrame
pca_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
X_pca_df = pd.DataFrame(X_pca, columns=pca_cols)

print("--- First 5 rows of PCA features ---")
print(X_pca_df.head())

--- First 5 rows of PCA features ---
        PC1       PC2       PC3       PC4       PC5       PC6       PC7  \
0  3.161988 -0.535368  1.018382  2.527486  1.140529  0.018784  1.214731   
1  2.708270 -0.917055  0.914819  2.449819  0.936151  0.272127  1.735774   
2  4.008291 -0.164348  0.821459  2.766427  1.343781 -0.110714  1.129284   
3  2.415157 -1.496226  0.650934  1.580681  1.506511  0.243886  0.137226   
4  3.340954 -0.583810  0.740874  2.746923  1.069663 -0.032296  1.674552   

        PC8       PC9  
0 -1.944898 -0.871711  
1 -1.297188 -2.285181  
2 -2.115737 -0.297025  
3 -1.335275 -1.206765  
4 -1.528435 -1.693298  


In [110]:
# Explained variance ratio for each component
explained_var = pca.explained_variance_ratio_
print("Explained Variance Ratio per component:")
print(explained_var)

# Cumulative variance retained
cumulative_var = np.cumsum(explained_var)
print("\nCumulative Variance Retained:")
print(cumulative_var)

Explained Variance Ratio per component:
[0.19296491 0.15752719 0.10698967 0.09046721 0.08418652 0.07840569
 0.07187061 0.06882404 0.06210583]

Cumulative Variance Retained:
[0.19296491 0.3504921  0.45748177 0.54794898 0.63213549 0.71054119
 0.7824118  0.85123584 0.91334168]
